<h1>Cropping</h1>

In [ ]:
#FOR TESTING

import json
import os
import cv2

# data path
image_dir = 'train/image'
anno_dir = 'train/annos'
output_dir = 'stage2_crops'
empty = [0, 0, 0, 0]

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for json_file in os.listdir(anno_dir):
    if not json_file.endswith('.json'):
        continue
        
    # Read the json
    with open(os.path.join(anno_dir, json_file), 'r') as f:
        data = json.load(f)
    
    # Load the corresponding image
    img_name = json_file.replace('.json', '.jpg')
    image = cv2.imread(os.path.join(image_dir, img_name))
    
    if image is None: continue

    # Read through the items
    for key in data.keys():
        if 'item' in key:
            item = data[key]
            bbox = item['bounding_box']     # [x1, y1, x2, y2]

            if (bbox == empty):
                continue
            
            x1, y1, x2, y2 = bbox
            
            crop_img = image[int(y1):int(y2), int(x1):int(x2)]
            
            crop_name = f"{img_name.split('.')[0]}_{key}.jpg"
            cv2.imwrite(os.path.join(output_dir, crop_name), crop_img)

print("success")

In [3]:
## FOR VALIDATION
import json
import os
import cv2

# data path
image_dir = 'val/images'
anno_dir = 'val/annos'
output_dir = 'stage2_crops_val'
empty = [0, 0, 0, 0]

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for json_file in os.listdir(anno_dir):
    if not json_file.endswith('.json'):
        continue
        
    # Read the json
    with open(os.path.join(anno_dir, json_file), 'r') as f:
        data = json.load(f)
    
    # Load the corresponding image
    img_name = json_file.replace('.json', '.jpg')
    image = cv2.imread(os.path.join(image_dir, img_name))
    
    if image is None: continue

    # Read through the items
    for key in data.keys():
        if 'item' in key:
            item = data[key]
            bbox = item['bounding_box']     # [x1, y1, x2, y2]

            if (bbox == empty):
                continue
            
            x1, y1, x2, y2 = bbox
            
            crop_img = image[int(y1):int(y2), int(x1):int(x2)]
            
            crop_name = f"{img_name.split('.')[0]}_{key}.jpg"
            cv2.imwrite(os.path.join(output_dir, crop_name), crop_img)

print("success")

success


<h1>Classifying</h1>

In [13]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [14]:
import os
import json
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torch import optim
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from PIL import Image
import time



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")



class CropDataSet(Dataset):
    def __init__(self, image_dir, json_dir, transform=None, max_samples = None):
        self.image_dir = image_dir
        self.json_dir = json_dir
        self.transform = transform
        self.all_imgs = sorted([f for f in os.listdir(image_dir)])
        if max_samples is not None:
            self.all_imgs = self.all_imgs[:max_samples]

    # Return the JSON of the image
    def get_json(self, image_str):
        json_path = os.path.join(self.json_dir, f"{image_str}.json")
        with open(json_path, 'r') as f:
            data = json.load(f)
        return data

    def __len__(self):
        return len(self.all_imgs)
    
    def __getitem__(self, idx):
        image_name = self.all_imgs[idx]
        image_id = image_name.split('_')[0]
        n_item = image_name.split('_')[1].split('.')[0]
        img_path = os.path.join(self.image_dir, image_name)
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        
        json_data = self.get_json(image_id)
        category = json_data[n_item]["category_id"]-1 #1(1-1=0) to 13(13-1=12), 0 is a id
        
        # sleeve = 1 if category in [2, 4, 11] else 0
        # target_type = 1 if category >= 7 else 0
        return img, category
        # return img, sleeve, target_type

Device: cpu


In [ ]:
class CropModel(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for param in backbone.parameters():
            param.requires_grad = False
        for param in backbone.layer4.parameters():
            param.requires_grad = True # Freezes all layers except layer4

        # extracts final feature layer before class score in ResNet
        self.feature_layers = nn.Sequential(*list(backbone.children())[:-1])
        
        # Custom Layouts According to previous papers
        self.fc1 = nn.Linear(2048, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.dropout2 = nn.Dropout(p=0.3)

        # Category classifier
        self.category_classifier = nn.Linear(512, 13)
    
    def forward(self, x):

        #ResNet50
        tensor = self.feature_layers(x)      # Shape: [batch, 2048, 1, 1]
        x = torch.flatten(tensor, 1)         # Shape: [batch, 2048]
        
        # Custom Layout according to previous papers
        x = self.fc1(x)                      # [batch, 1024]
        x = self.bn1(x)                      # [batch, 1024]
        x = self.relu(x)                     # [batch, 1024]
        x = self.dropout1(x)                 # [batch, 1024]
        
        x = self.fc2(x)                      # [batch, 512]
        x = self.bn2(x)                      # [batch, 512]
        x = self.relu(x)                     # [batch, 512]
        x = self.dropout2(x)                 # [batch, 512]
        
        category_out = self.category_classifier(x)  # [batch, 13]
        return category_out

In [18]:
import os
os.getcwd()

'/Users/vas/Desktop/UniversityDocuments/COMP4471/project/project-classification/comp4471project_group29'

In [19]:

image_dir = "stage2_crops"
json_dir = "train/annos"
val_image_dir = "stage2_crops_val"
val_json_dir = "val/annos"
batch_size = 16
num_epochs = 3
learning_rate = 0.001

transform = transforms.Compose([
transforms.Resize((224, 224)),
transforms.ToTensor(),
transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

datasets = CropDataSet(image_dir, json_dir, transform=transform, max_samples=20000)
dataloader = DataLoader(datasets, batch_size=batch_size, shuffle=True)

val_datasets = CropDataSet(val_image_dir, val_json_dir, transform=transform, max_samples=20000)
val_loader = DataLoader(val_datasets, batch_size=batch_size, shuffle=False)

model = CropModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

for i in range(num_epochs):
    start = time.time()
    print(f"Epoch [{i+1}]")


    # Training
    model.train()
    total_loss = 0
    for images, category_labels in dataloader:
        images = images.to(device)
        category_labels = category_labels.long().to(device)

        optimizer.zero_grad()
        category_out = model(images)
        loss_category = criterion(category_out, category_labels)
        loss = loss_category
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"time = {time.time()-start}")
    print(f"Epoch [{i+1}/{num_epochs}], Loss: {avg_loss:.4f}")
    # Validation Score Per Epoch
    # After training loop, add evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:  # Use a validation loader
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Accuracy: {100 * correct / total:.2f}%")


torch.save(model.state_dict(), 'classification_model.pth')
print("Model saved in local drive")

Epoch [1]
time = 23.9055118560791
Epoch [1/3], Loss: 1.4369
Accuracy: 23.53%
Epoch [2]
time = 24.082586765289307
Epoch [2/3], Loss: 0.5686
Accuracy: 25.74%
Epoch [3]
time = 24.574373960494995
Epoch [3/3], Loss: 0.3042
Accuracy: 28.68%
Model saved in local drive
